# 📘 Module 7.1 — Vision-Language-Action (VLA) Models

**Goal:** Understand VLA models — the frontier of end-to-end autonomous driving.

## What are VLAs?

VLAs extend VLMs by adding **action prediction** — they don't just understand scenes, they decide what to DO.

```
VLM:  Image + Text → Understanding ("There's a pedestrian crossing")
VLA:  Image + Text → Understanding + Action ("Brake and yield")
```

### The Evolution
```
CNN (see) → VLM (see + understand) → VLA (see + understand + act)
```

### Key VLA Models for Autonomous Driving
| Model | Organization | Key Innovation |
|-------|-------------|----------------|
| **RT-2** | Google DeepMind | Robotic Transformer with VLM |
| **DriveVLM** | Various | VLM-based driving policy |
| **LMDrive** | Research | Language-guided driving |
| **OpenVLA** | Stanford | Open-source VLA |
| **EMMA** | Waymo | End-to-end multimodal model |

---

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

## 1. VLA Architecture for Driving

```
Multi-Camera Images ──► Vision Encoder ──► Visual Tokens ─┐
                                                            │
Navigation Command ───► Text Encoder  ──► Text Tokens  ──┤──► Transformer ──► Actions
                                                            │         (steering, throttle,
Past Actions/State ───► State Encoder ──► State Tokens ──┘          brake, waypoints)
```

In [ ]:
class SimpleVLA(nn.Module):
    """
    Simplified Vision-Language-Action model for driving.
    Demonstrates the concept of multimodal input → action output.
    """
    
    def __init__(self, d_model=256, num_actions=4):
        super().__init__()
        
        # Vision encoder (simplified)
        self.vision_encoder = nn.Sequential(
            nn.Conv2d(3, 64, 7, stride=4, padding=3),
            nn.ReLU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((7, 7)),
            nn.Flatten(),
            nn.Linear(128 * 7 * 7, d_model),
        )
        
        # Language encoder (for navigation commands)
        self.text_embed = nn.Embedding(1000, d_model)
        self.text_encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model, nhead=4, dim_feedforward=512, batch_first=True),
            num_layers=2
        )
        
        # State encoder (ego vehicle state: speed, heading, etc.)
        self.state_encoder = nn.Sequential(
            nn.Linear(6, d_model),  # 6 state features
            nn.ReLU(),
        )
        
        # Fusion transformer
        self.fusion = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model, nhead=8, dim_feedforward=1024, batch_first=True),
            num_layers=4
        )
        
        # Action head
        # num_actions: [steering, throttle, brake, turn_signal]
        self.action_head = nn.Sequential(
            nn.Linear(d_model, 128),
            nn.ReLU(),
            nn.Linear(128, num_actions),
        )
        
        # Waypoint prediction head (future trajectory)
        self.waypoint_head = nn.Sequential(
            nn.Linear(d_model, 128),
            nn.ReLU(),
            nn.Linear(128, 10 * 2),  # 10 future waypoints (x, y)
        )
    
    def forward(self, image, text_tokens, ego_state):
        # Encode each modality
        visual_feat = self.vision_encoder(image).unsqueeze(1)  # (B, 1, d_model)
        
        text_feat = self.text_embed(text_tokens)
        text_feat = self.text_encoder(text_feat)  # (B, seq, d_model)
        text_feat = text_feat.mean(dim=1, keepdim=True)  # Pool to (B, 1, d_model)
        
        state_feat = self.state_encoder(ego_state).unsqueeze(1)  # (B, 1, d_model)
        
        # Fuse all modalities
        fused = torch.cat([visual_feat, text_feat, state_feat], dim=1)  # (B, 3, d_model)
        fused = self.fusion(fused)  # (B, 3, d_model)
        
        # Pool fused features
        pooled = fused.mean(dim=1)  # (B, d_model)
        
        # Predict actions and waypoints
        actions = self.action_head(pooled)  # (B, 4)
        waypoints = self.waypoint_head(pooled).view(-1, 10, 2)  # (B, 10, 2)
        
        return actions, waypoints

# Test
vla = SimpleVLA()
image = torch.randn(2, 3, 224, 224)
text = torch.randint(0, 1000, (2, 10))  # "Turn left at the intersection"
state = torch.tensor([[30.0, 0.0, 0.0, 1.0, 0.0, 0.0],   # speed, heading, etc.
                      [45.0, 0.1, 0.0, 1.0, 0.0, 0.0]])

actions, waypoints = vla(image, text, state)

print(f"Image: {image.shape}")
print(f"Text tokens: {text.shape}")
print(f"Ego state: {state.shape}")
print(f"Actions: {actions.shape}  [steering, throttle, brake, signal]")
print(f"Waypoints: {waypoints.shape}  [10 future (x,y) points]")
print(f"Parameters: {sum(p.numel() for p in vla.parameters()):,}")

In [ ]:
# --- Visualize predicted trajectory ---
vla.eval()
with torch.no_grad():
    actions, waypoints = vla(image[:1], text[:1], state[:1])

wp = waypoints[0].numpy()

fig, ax = plt.subplots(figsize=(8, 8))
# Plot predicted trajectory
ax.plot(wp[:, 0], wp[:, 1], 'b-o', markersize=8, linewidth=2, label='Predicted Path')
ax.plot(0, 0, 'g^', markersize=15, label='Ego Vehicle')
ax.arrow(0, 0, 0, 0.5, head_width=0.1, head_length=0.05, fc='green', ec='green')

ax.set_xlabel('Lateral (m)')
ax.set_ylabel('Longitudinal (m)')
ax.set_title('VLA Predicted Trajectory\n(random weights — not trained)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

print(f"Predicted actions:")
action_names = ['Steering', 'Throttle', 'Brake', 'Signal']
for name, val in zip(action_names, actions[0].numpy()):
    print(f"  {name}: {val:.4f}")

## 2. Language-Conditioned Driving

VLAs can follow **natural language instructions**:
- "Turn left at the next intersection"
- "Slow down, there's a school zone ahead"
- "Change to the right lane"

In [ ]:
# --- Language Commands for Driving ---
commands = [
    ("Follow the car ahead", "Maintain safe distance, match speed"),
    ("Take the next exit", "Signal right, decelerate, follow exit ramp"),
    ("Stop for the pedestrian", "Apply brakes, come to full stop, wait"),
    ("Park in the empty spot on the right", "Signal right, slow down, execute parking maneuver"),
    ("Avoid the obstacle ahead", "Check mirrors, signal, change lanes safely"),
]

print("Language-Conditioned Driving Actions:")
print("=" * 60)
for cmd, action in commands:
    print(f"  🗣️ Command: '{cmd}'")
    print(f"  🚗 Action:  {action}")
    print()

## 3. Challenges & Future of VLAs in ADAS

| Challenge | Description | Current Solutions |
|-----------|------------|-------------------|
| **Latency** | Must decide in <100ms | Model distillation, quantization |
| **Safety** | Wrong action = crash | Constrained action spaces, fallback systems |
| **Hallucination** | Model "sees" nonexistent objects | Grounded visual reasoning |
| **Sim-to-Real** | Training in simulation ≠ real world | Domain adaptation |
| **Interpretability** | Why did it decide to brake? | Attention visualization, chain-of-thought |

In [ ]:
print("🔮 The Future of VLAs in Autonomous Driving:")
print()
print("Current State (2024-2026):")
print("  • Research prototypes showing promising results")
print("  • Waymo's EMMA model integrating multimodal understanding")
print("  • Tesla using transformer-based end-to-end models")
print()
print("Near Future:")
print("  • VLAs handling complex negotiations (merging, yielding)")
print("  • Natural language explanations of driving decisions")
print("  • Continuous learning from fleet data")
print()
print("Long Term:")
print("  • Fully autonomous VLA-powered driving")
print("  • Conversational co-pilots that explain and learn")
print("  • World models that predict and plan in imagination")

---
## ✅ Key Takeaways

1. **VLAs** = Vision + Language + Action — they see, understand, and act
2. VLAs use **multimodal fusion** of camera, language, and vehicle state
3. **Language conditioning** enables flexible driving commands
4. Key outputs: steering, throttle, brake, and future trajectory waypoints
5. Challenges: latency, safety guarantees, and sim-to-real transfer

---
## 📖 Next Steps
➡️ **Next notebook:** [02_end_to_end_driving.ipynb](02_end_to_end_driving.ipynb) — End-to-end driving models